In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install alpaca-trade-api

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Multi-asset 1-hour bars via Alpaca
# For Colab: run setup_colab_secrets() first if needed

TICKERS = ['QQQ', 'SPY', 'NVDA', 'AAPL', 'MSFT', 'AMD', 'META', 'AMZN']
PRIMARY_TICKER = 'QQQ'
START_DATE = '2024-01-01'

# Download hourly data (Alpaca provides 1h bars with VWAP)
all_data = download_multi(TICKERS, START_DATE, timeframe='1h')

# Primary ticker for grid search
stock_data = download(PRIMARY_TICKER, START_DATE, timeframe='1h')
TICKER = PRIMARY_TICKER

if not stock_data.empty:
    print(f"Downloaded {len(stock_data)} hourly bars for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min()} to {stock_data.index.max()}")
    print(f"Columns: {list(stock_data.columns)}")
    print(f"\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} hourly data")

stock_data

In [ ]:
# COMPUTE INTRADAY INDICATORS ON HOURLY BARS
# - Cumulative VWAP (resetting each day)
# - RSI(2) on hourly closes
# - ATR(14) on hourly OHLC
# - Daily open for trend direction

if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].astype(float).squeeze()
    high  = stock_data[("High",  TICKER)].astype(float).squeeze()
    low   = stock_data[("Low",   TICKER)].astype(float).squeeze()
    open_ = stock_data[("Open",  TICKER)].astype(float).squeeze()
    volume = stock_data[("Volume", TICKER)].astype(float).squeeze()
else:
    close  = stock_data["Close"].astype(float).squeeze()
    high   = stock_data["High"].astype(float).squeeze()
    low    = stock_data["Low"].astype(float).squeeze()
    open_  = stock_data["Open"].astype(float).squeeze()
    volume = stock_data["Volume"].astype(float).squeeze()

# --- Cumulative session VWAP (reset each trading day) ---
# Even if Alpaca provides a VWAP column, it's bar-level, not cumulative.
# We compute cumulative VWAP ourselves: cumsum(close*volume) / cumsum(volume) per day.
dates = close.index.date
unique_dates = pd.Series(dates).unique()

vwap = pd.Series(np.nan, index=close.index, dtype=float)
daily_open = pd.Series(np.nan, index=close.index, dtype=float)

for d in unique_dates:
    mask = dates == d
    day_close = close[mask].values
    day_vol   = volume[mask].values

    # Cumulative VWAP for this session
    cum_pv = np.cumsum(day_close * day_vol)
    cum_v  = np.cumsum(day_vol)
    day_vwap = np.where(cum_v > 0, cum_pv / cum_v, day_close)
    vwap[mask] = day_vwap

    # Daily open = first bar's open
    daily_open[mask] = open_[mask].iloc[0]

# RSI on hourly closes (default period will be parameterized)
rsi_2  = pd.Series(talib.RSI(close.values, timeperiod=2), index=close.index)
rsi_3  = pd.Series(talib.RSI(close.values, timeperiod=3), index=close.index)
rsi_5  = pd.Series(talib.RSI(close.values, timeperiod=5), index=close.index)

# ATR on hourly OHLC
atr_14 = pd.Series(talib.ATR(high.values, low.values, close.values, timeperiod=14), index=close.index)

print(f"Indicators computed on {len(close)} hourly bars")
print(f"  VWAP range: {vwap.min():.2f} - {vwap.max():.2f}")
print(f"  RSI(2) range: {rsi_2.dropna().min():.1f} - {rsi_2.dropna().max():.1f}")
print(f"  ATR(14) range: {atr_14.dropna().min():.4f} - {atr_14.dropna().max():.4f}")
print(f"  Unique trading days: {len(unique_dates)}")

In [ ]:
# PREPARE PRICE SERIES + IN-SAMPLE / OUT-OF-SAMPLE SPLIT (60/40)

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

close_series = close.copy()
close_series.name = 'price'

TRAIN_RATIO = 0.60
split_idx = int(len(close_series) * TRAIN_RATIO)

train_close = close_series.iloc[:split_idx].copy()
val_close   = close_series.iloc[split_idx:].copy()

# Also split all indicator series
train_high = high.iloc[:split_idx].copy()
val_high   = high.iloc[split_idx:].copy()
train_low  = low.iloc[:split_idx].copy()
val_low    = low.iloc[split_idx:].copy()
train_open = open_.iloc[:split_idx].copy()
val_open   = open_.iloc[split_idx:].copy()
train_volume = volume.iloc[:split_idx].copy()
val_volume   = volume.iloc[split_idx:].copy()

# Pre-compute RSI for all periods
rsi_cache = {
    2: pd.Series(talib.RSI(close.values, timeperiod=2), index=close.index),
    3: pd.Series(talib.RSI(close.values, timeperiod=3), index=close.index),
    5: pd.Series(talib.RSI(close.values, timeperiod=5), index=close.index),
}

print(f"Data ready:")
print(f"  Train: {train_close.index[0]} -> {train_close.index[-1]} ({len(train_close)} bars)")
print(f"  Val:   {val_close.index[0]} -> {val_close.index[-1]} ({len(val_close)} bars)")

VWAP Mean Reversion Strategy — Intraday Hourly
================================================

**Core Thesis**: VWAP acts as the intraday "fair value" anchor. When price deviates from VWAP and then reverts, it creates high-probability mean reversion entries.

**Entry Logic (Long-Only)**:
1. Price was **above VWAP** in the previous bar (established above fair value)
2. Price pulls back to **within ATR distance of VWAP** (mean reversion touch)
3. **RSI(2)** < oversold threshold (confirms short-term oversold on pullback)
4. **Trend filter**: price > daily open (bullish session bias)

**Exit Logic**:
1. **Trailing stop**: entry price - (ATR_stop_mult x ATR)
2. **Profit target**: entry price + (target_mult x ATR)
3. **End of day**: close all positions at last bar of session

**Anti-Lookahead**: All indicator values used at bar i come from bar i-1 or earlier.

**Key Parameters**:
- `rsi_period`: RSI lookback (2, 3, 5)
- `rsi_oversold`: Oversold threshold (10, 20, 30)
- `atr_stop_mult`: ATR multiplier for stop loss (1.0 - 2.5)
- `target_mult`: ATR multiplier for profit target (1.5 - 3.0)

---

In [ ]:
def generate_vwap_mr_signals(hourly_close, hourly_high, hourly_low, hourly_open,
                             hourly_volume, rsi_period=2, rsi_oversold=20,
                             rsi_overbought=80, atr_stop_mult=1.5, target_mult=2.0,
                             require_trend_filter=True):
    """
    VWAP Mean Reversion on hourly bars (long-only for simplicity).

    Entry conditions (all must be true, using PREVIOUS bar values):
    1. Price was ABOVE VWAP in previous bar (established above fair value)
    2. Price pulls back to within 1 ATR of VWAP (mean reversion zone)
    3. RSI(rsi_period) < rsi_oversold (oversold on pullback)
    4. If trend filter: price > daily open (bullish session)

    Exit conditions:
    1. Trailing stop: entry - atr_stop_mult * ATR
    2. Target: entry + target_mult * ATR
    3. End of day (last bar of session)

    All signals use previous bar values (anti-lookahead).
    Returns: entries (bool Series), exits (bool Series)
    """
    idx = hourly_close.index
    n = len(hourly_close)

    # --- Compute indicators ---
    close_vals = hourly_close.values.astype(float)
    high_vals  = hourly_high.values.astype(float)
    low_vals   = hourly_low.values.astype(float)
    open_vals  = hourly_open.values.astype(float)
    vol_vals   = hourly_volume.values.astype(float)

    # RSI
    if rsi_period in rsi_cache:
        rsi_full = rsi_cache[rsi_period]
        rsi_vals = rsi_full.reindex(idx).values
    else:
        rsi_vals = talib.RSI(close_vals, timeperiod=rsi_period)

    # ATR(14) on hourly
    atr_vals = talib.ATR(high_vals, low_vals, close_vals, timeperiod=14)

    # Cumulative session VWAP (reset daily)
    dates = pd.Series(idx.date)
    vwap_vals = np.full(n, np.nan)
    daily_open_vals = np.full(n, np.nan)
    is_last_bar = np.zeros(n, dtype=bool)

    unique_dates = dates.unique()
    for d in unique_dates:
        mask = (dates == d).values
        day_indices = np.where(mask)[0]

        if len(day_indices) == 0:
            continue

        # Cumulative VWAP
        day_close = close_vals[day_indices]
        day_vol   = vol_vals[day_indices]
        cum_pv = np.cumsum(day_close * day_vol)
        cum_v  = np.cumsum(day_vol)
        day_vwap = np.where(cum_v > 0, cum_pv / cum_v, day_close)
        vwap_vals[day_indices] = day_vwap

        # Daily open
        daily_open_vals[day_indices] = open_vals[day_indices[0]]

        # Last bar of day
        is_last_bar[day_indices[-1]] = True

    # --- Bar-by-bar signal generation ---
    entries = np.zeros(n, dtype=bool)
    exits   = np.zeros(n, dtype=bool)

    in_trade = False
    entry_price = 0.0
    stop_price  = 0.0
    target_price = 0.0
    highest_since_entry = 0.0

    for i in range(2, n):  # start at 2 to have prev bar values
        # Use PREVIOUS bar values for signals (anti-lookahead)
        prev_close = close_vals[i - 1]
        prev_vwap  = vwap_vals[i - 1]
        prev_rsi   = rsi_vals[i - 1]
        prev_atr   = atr_vals[i - 1]
        prev_daily_open = daily_open_vals[i - 1]
        prev2_close = close_vals[i - 2]
        prev2_vwap  = vwap_vals[i - 2]

        if np.isnan(prev_atr) or np.isnan(prev_rsi) or np.isnan(prev_vwap):
            continue

        if in_trade:
            # Check exit conditions using current bar
            current_close = close_vals[i]

            # Update trailing stop (highest since entry)
            if current_close > highest_since_entry:
                highest_since_entry = current_close

            # Exit 1: Stop loss hit
            if current_close <= stop_price:
                exits[i] = True
                in_trade = False
                continue

            # Exit 2: Target hit
            if current_close >= target_price:
                exits[i] = True
                in_trade = False
                continue

            # Exit 3: End of day
            if is_last_bar[i]:
                exits[i] = True
                in_trade = False
                continue

        else:
            # Check entry conditions (all using previous bar)
            # 1. Two bars ago price was ABOVE VWAP (was above fair value)
            cond_above_vwap = prev2_close > prev2_vwap

            # 2. Previous bar price pulled back to within 1 ATR of VWAP
            dist_to_vwap = abs(prev_close - prev_vwap)
            cond_near_vwap = dist_to_vwap <= prev_atr

            # 3. RSI oversold
            cond_rsi = prev_rsi < rsi_oversold

            # 4. Trend filter: price > daily open (bullish session)
            if require_trend_filter:
                cond_trend = prev_close > prev_daily_open
            else:
                cond_trend = True

            if cond_above_vwap and cond_near_vwap and cond_rsi and cond_trend:
                entries[i] = True
                in_trade = True
                entry_price = close_vals[i]  # enter at this bar's close
                stop_price  = entry_price - atr_stop_mult * prev_atr
                target_price = entry_price + target_mult * prev_atr
                highest_since_entry = entry_price

    return pd.Series(entries, index=idx, dtype=bool), pd.Series(exits, index=idx, dtype=bool)


# Quick test with default params
test_e, test_x = generate_vwap_mr_signals(
    train_close, train_high, train_low, train_open, train_volume,
    rsi_period=2, rsi_oversold=20, atr_stop_mult=1.5, target_mult=2.0
)
print(f"Test signals: {test_e.sum()} entries, {test_x.sum()} exits on training data")

In [ ]:
# PARAMETER RANGES FOR GRID SEARCH

rsi_period_range  = [2, 3, 5]
rsi_oversold_range = [10, 20, 30]
atr_stop_range    = [1.0, 1.5, 2.0, 2.5]
target_range      = [1.5, 2.0, 2.5, 3.0]

# Generate all combinations
param_combinations = list(product(rsi_period_range, rsi_oversold_range,
                                   atr_stop_range, target_range))

print(f"RSI Periods:      {rsi_period_range}")
print(f"RSI Oversold:     {rsi_oversold_range}")
print(f"ATR Stop Mult:    {atr_stop_range}")
print(f"Target Mult:      {target_range}")
print(f"\nTotal combinations: {len(param_combinations)}")
print(f"\nFirst 10 combinations:")
for i, (rp, ro, asm, tm) in enumerate(param_combinations[:10], 1):
    print(f"  {i:2d}. RSI({rp}) OS={ro}, Stop={asm}x, Target={tm}x")
if len(param_combinations) > 10:
    print(f"  ... and {len(param_combinations) - 10} more")

In [ ]:
# INITIALIZE RESULTS COLLECTION

grid_search_results = []

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'h'  # hourly frequency for vectorbt

metrics_to_collect = [
    "rsi_period", "rsi_oversold", "atr_stop_mult", "target_mult",
    "total_return", "annualized_return",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio", "omega_ratio",
    "max_drawdown", "volatility", "ulcer_index",
    "win_rate", "total_trades", "expectancy",
    "profit_factor", "sqn",
    "payoff_ratio", "largest_win", "largest_loss",
    "avg_win_amount", "avg_loss_amount", "winning_streak", "losing_streak",
    "recovery_factor", "gain_to_pain_ratio", "serenity_index"
]

print(f"Results system initialized")
print(f"  Init cash: ${INIT_CASH:,}")
print(f"  Fees: {FEES*100:.2f}%  |  Slippage: {SLIPPAGE*100:.2f}%")
print(f"  Frequency: hourly")
print(f"  {len(metrics_to_collect)} metrics per combination")

In [ ]:
# VWAP MEAN REVERSION GRID SEARCH ON TRAINING DATA

print("INITIATING VWAP MEAN REVERSION GRID SEARCH")
print("=" * 70)
print(f"Strategy: VWAP Mean Reversion (Long-Only, Hourly)")
print(f"Ticker: {TICKER}")
print(f"Training Period: {train_close.index[0]} -> {train_close.index[-1]}")
print(f"Initial Capital: ${INIT_CASH:,}")
print(f"Fees + Slippage: {(FEES+SLIPPAGE)*100:.2f}% per trade")
print("=" * 70)

total_combinations = len(param_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing with progress every 50 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_hours = max((train_close.index[-1] - train_close.index[0]).total_seconds() / 3600, 1)
train_years = train_hours / (252 * 6.5)  # approx trading hours per year

for combo_num, (rsi_p, rsi_os, atr_s, tgt) in enumerate(param_combinations, 1):
    try:
        entries, exits = generate_vwap_mr_signals(
            train_close, train_high, train_low, train_open, train_volume,
            rsi_period=rsi_p, rsi_oversold=rsi_os,
            atr_stop_mult=atr_s, target_mult=tgt,
            require_trend_filter=True
        )

        if entries.sum() < 5:
            skipped_low_trades += 1
            continue

        pf = vbt.Portfolio.from_signals(
            close=train_close, entries=entries, exits=exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
        )

        trades = pf.trades
        total_trades = len(trades)

        if total_trades < 5:
            skipped_low_trades += 1
            continue

        total_return = float(pf.total_return())
        try:
            annualized_return = float(pf.annualized_return(freq=FREQ))
        except:
            annualized_return = np.nan
        max_drawdown = float(pf.max_drawdown())
        try:
            volatility = float(pf.annualized_volatility(freq=FREQ))
        except:
            volatility = np.nan
        try:
            sharpe_ratio = float(pf.sharpe_ratio(freq=FREQ))
        except:
            sharpe_ratio = np.nan
        try:
            sortino_ratio = float(pf.sortino_ratio(freq=FREQ))
        except:
            sortino_ratio = np.nan
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan

        tr = np.asarray(trades.returns.values if hasattr(trades.returns, 'values') else trades.returns).ravel()
        pnl = np.asarray(trades.pnl.values if hasattr(trades.pnl, 'values') else trades.pnl).ravel()

        pos = tr[tr > 0]; neg = tr[tr < 0]
        win_rate = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
        profit_factor = pos.sum() / abs(neg.sum()) if len(neg) > 0 and abs(neg.sum()) > 0 else np.inf
        expectancy = float(tr.mean()) if len(tr) > 0 else 0.0
        avg_win = float(pos.mean()) if len(pos) > 0 else 0.0
        avg_loss = float(abs(neg.mean())) if len(neg) > 0 else 0.0
        largest_win = float(pos.max()) if len(pos) > 0 else 0.0
        largest_loss = float(neg.min()) if len(neg) > 0 else 0.0
        sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if len(tr) > 1 and tr.std() > 0 else np.nan
        payoff_ratio = avg_win / avg_loss if avg_loss > 0 else np.inf

        returns = pf.returns()
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        cum = (1 + returns).cumprod(); peak = cum.cummax(); dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan

        winning_streak = np.nan; losing_streak = np.nan
        try:
            winning_streak = int(trades.winning_streak())
            losing_streak = int(trades.losing_streak())
        except: pass

        grid_search_results.append({
            "rsi_period": rsi_p, "rsi_oversold": rsi_os,
            "atr_stop_mult": atr_s, "target_mult": tgt,
            "total_return": total_return, "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio, "sortino_ratio": sortino_ratio,
            "calmar_ratio": calmar_ratio, "omega_ratio": omega_ratio,
            "max_drawdown": max_drawdown, "volatility": volatility,
            "ulcer_index": ulcer_index, "win_rate": win_rate,
            "total_trades": total_trades, "expectancy": expectancy,
            "profit_factor": profit_factor, "sqn": sqn,
            "payoff_ratio": payoff_ratio, "largest_win": largest_win,
            "largest_loss": largest_loss, "avg_win_amount": avg_win,
            "avg_loss_amount": avg_loss, "winning_streak": winning_streak,
            "losing_streak": losing_streak, "recovery_factor": recovery_factor,
            "gain_to_pain_ratio": gain_to_pain, "serenity_index": serenity_index
        })
        successful_tests += 1

    except Exception as ex:
        failed_tests += 1

    if combo_num % 50 == 0 or combo_num == total_combinations:
        print(f"  Progress: {combo_num}/{total_combinations} | Valid: {successful_tests} | Skipped(<5 trades): {skipped_low_trades} | Failed: {failed_tests}")

print(f"\n{'='*70}")
print(f"GRID SEARCH COMPLETE")
print(f"  Tested: {total_combinations} | Valid: {successful_tests} | Skipped: {skipped_low_trades} | Failed: {failed_tests}")

if grid_search_results:
    results_df = pd.DataFrame(grid_search_results)
    top10 = results_df.nlargest(10, 'sharpe_ratio')
    print(f"\nTOP 10 BY SHARPE RATIO (Training Set):")
    print("=" * 90)
    for rank, (_, row) in enumerate(top10.iterrows(), 1):
        print(f"  #{rank:2d}  RSI({int(row['rsi_period'])}) OS={int(row['rsi_oversold'])} "
              f"Stop={row['atr_stop_mult']:.1f}x Tgt={row['target_mult']:.1f}x  |  "
              f"Sharpe={row['sharpe_ratio']:.3f}  Return={row['total_return']:.2%}  "
              f"MaxDD={row['max_drawdown']:.2%}  Trades={int(row['total_trades'])}  "
              f"WR={row['win_rate']:.1f}%  PF={row['profit_factor']:.2f}")
else:
    print("No valid results found. Check data and parameters.")
    results_df = pd.DataFrame()

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & EQUITY CURVES

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period:    {train_close.index[0]} -> {train_close.index[-1]}")
    print(f"Validation Period:  {val_close.index[0]} -> {val_close.index[-1]}")
    print("=" * 90)

    top_5 = results_df.nlargest(5, 'sharpe_ratio')
    oos_results = []

    for _, strategy in top_5.iterrows():
        rp = int(strategy['rsi_period'])
        ro = int(strategy['rsi_oversold'])
        asm = float(strategy['atr_stop_mult'])
        tm  = float(strategy['target_mult'])

        e_oos, x_oos = generate_vwap_mr_signals(
            val_close, val_high, val_low, val_open, val_volume,
            rsi_period=rp, rsi_oversold=ro,
            atr_stop_mult=asm, target_mult=tm,
            require_trend_filter=True
        )

        pf_val = vbt.Portfolio.from_signals(
            close=val_close, entries=e_oos, exits=x_oos,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
        )

        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ)) if len(pf_val.trades) > 0 else np.nan
        oos_ret    = float(pf_val.total_return())
        oos_dd     = float(pf_val.max_drawdown())
        oos_trades = len(pf_val.trades)

        oos_wr = np.nan; oos_pf = np.nan
        if oos_trades > 0:
            tr_arr = np.asarray(pf_val.trades.returns.values if hasattr(pf_val.trades.returns, 'values') else pf_val.trades.returns).ravel()
            if tr_arr.size > 0:
                p = tr_arr[tr_arr > 0]; n_arr = tr_arr[tr_arr < 0]
                oos_wr = (len(p)/len(tr_arr))*100 if len(tr_arr) > 0 else np.nan
                oos_pf = p.sum()/abs(n_arr.sum()) if len(n_arr) > 0 and abs(n_arr.sum()) > 0 else np.inf

        is_sharpe = float(strategy['sharpe_ratio'])
        sharpe_deg = ((oos_sharpe - is_sharpe) / abs(is_sharpe) * 100) if is_sharpe != 0 and not np.isnan(oos_sharpe) else np.nan

        oos_results.append({
            'Rank': len(oos_results)+1,
            'rsi_period': rp, 'rsi_oversold': ro,
            'atr_stop_mult': asm, 'target_mult': tm,
            'IS_Sharpe': is_sharpe, 'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']), 'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe, 'OOS_Return': oos_ret, 'OOS_MaxDD': oos_dd,
            'OOS_WinRate': oos_wr, 'OOS_Trades': oos_trades, 'OOS_ProfitFactor': oos_pf,
            'Sharpe_Degradation': sharpe_deg,
        })

    oos_df = pd.DataFrame(oos_results).sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)
    oos_df['OOS_Rank'] = range(1, len(oos_df)+1)

    print("\nCOMPREHENSIVE COMPARISON TABLE (Sorted by OOS Sharpe)")
    print("=" * 90)
    display_df = pd.DataFrame({
        'IS->OOS': oos_df['Rank'].astype(str) + '->' + oos_df['OOS_Rank'].astype(str),
        'Strategy': oos_df.apply(lambda x: f"VWAP_MR(RSI{int(x['rsi_period'])} OS{int(x['rsi_oversold'])} S{x['atr_stop_mult']:.1f} T{x['target_mult']:.1f})", axis=1),
        'IS Sharpe': oos_df['IS_Sharpe'].map('{:.3f}'.format),
        'OOS Sharpe': oos_df['OOS_Sharpe'].map('{:.3f}'.format),
        'Sharpe D%': oos_df['Sharpe_Degradation'].map('{:+.1f}%'.format),
        'IS Return': oos_df['IS_Return'].map('{:.1%}'.format),
        'OOS Return': oos_df['OOS_Return'].map('{:.1%}'.format),
        'OOS Trades': oos_df['OOS_Trades'].astype(int),
        'OOS WR': oos_df['OOS_WinRate'].map('{:.1f}%'.format),
        'OOS PF': oos_df['OOS_ProfitFactor'].map('{:.2f}'.format),
    })
    print(display_df.to_string(index=False))

    bo = oos_df.iloc[0]
    print(f"\nBEST OOS PERFORMER: VWAP_MR(RSI{int(bo['rsi_period'])} OS{int(bo['rsi_oversold'])} "
          f"S{bo['atr_stop_mult']:.1f} T{bo['target_mult']:.1f})")
    print(f"  OOS Sharpe: {bo['OOS_Sharpe']:.3f}  |  OOS Return: {bo['OOS_Return']:.2%}  |  "
          f"OOS MaxDD: {bo['OOS_MaxDD']:.2%}  |  Sharpe Degradation: {bo['Sharpe_Degradation']:+.1f}%")

    # --- Equity Curves for Best Strategy ---
    best_is = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    bp = {
        'rsi_period': int(best_is['rsi_period']),
        'rsi_oversold': int(best_is['rsi_oversold']),
        'atr_stop_mult': float(best_is['atr_stop_mult']),
        'target_mult': float(best_is['target_mult']),
    }

    # IS equity
    e_is, x_is = generate_vwap_mr_signals(
        train_close, train_high, train_low, train_open, train_volume, **bp)
    pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

    # OOS equity
    e_oos, x_oos = generate_vwap_mr_signals(
        val_close, val_high, val_low, val_open, val_volume, **bp)
    pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                         init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

    # Full sample
    e_full, x_full = generate_vwap_mr_signals(
        close_series, high, low, open_, volume, **bp)
    pf_full = vbt.Portfolio.from_signals(close=close_series, entries=e_full, exits=x_full,
                                          init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

    # Buy & hold
    bh_e = pd.Series(False, index=close_series.index, dtype=bool); bh_e.iloc[0] = True
    bh_x = pd.Series(False, index=close_series.index, dtype=bool)
    pf_bh = vbt.Portfolio.from_signals(close=close_series, entries=bh_e, exits=bh_x,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f'VWAP Mean Reversion - {TICKER} | RSI({bp["rsi_period"]}) OS={bp["rsi_oversold"]} '
                 f'Stop={bp["atr_stop_mult"]}x Target={bp["target_mult"]}x', fontsize=14, fontweight='bold')

    eq = pf_full.value(); eq_bh = pf_bh.value()
    ax1.plot(close_series.index[:split_idx], eq.iloc[:split_idx], color='#2563EB', lw=2, label='Strategy (IS)')
    ax1.plot(close_series.index[split_idx:], eq.iloc[split_idx:], color='#EA580C', lw=2, label='Strategy (OOS)')
    ax1.plot(close_series.index, eq_bh, color='gray', lw=1.2, ls='--', alpha=0.6, label='Buy & Hold')
    ax1.axvline(x=close_series.index[split_idx], color='red', ls=':', alpha=0.3)
    ax1.set_ylabel('Portfolio Value ($)'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    dd = pf_full.drawdown() * 100
    ax2.fill_between(close_series.index, dd.values, 0, color='red', alpha=0.2)
    ax2.plot(close_series.index, dd.values, color='red', lw=0.8, alpha=0.7)
    ax2.set_ylabel('Drawdown %'); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

PARAMETER SENSITIVITY ANALYSIS
------------------------------

For each parameter (`rsi_period`, `rsi_oversold`, `atr_stop_mult`, `target_mult`), we vary it across its range while holding the other three fixed at their best values.

**Color coding**: Dark green (>+10% vs base), Light green (0-10%), Orange (-10 to 0%), Red (<-10%).

---

In [ ]:
# PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    base_rp  = int(best['rsi_period'])
    base_ros = int(best['rsi_oversold'])
    base_asm = float(best['atr_stop_mult'])
    base_tm  = float(best['target_mult'])
    base_is_sharpe = float(best['sharpe_ratio'])

    # Compute base OOS sharpe
    base_oos_sharpe = np.nan
    try:
        e_c, x_c = generate_vwap_mr_signals(
            val_close, val_high, val_low, val_open, val_volume,
            rsi_period=base_rp, rsi_oversold=base_ros,
            atr_stop_mult=base_asm, target_mult=base_tm)
        pf_c = vbt.Portfolio.from_signals(close=val_close, entries=e_c, exits=x_c,
                                           init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
        base_oos_sharpe = float(pf_c.sharpe_ratio(freq=FREQ))
    except: pass

    print(f"SENSITIVITY ANALYSIS")
    print(f"Base: RSI({base_rp}) OS={base_ros} Stop={base_asm}x Target={base_tm}x")
    print(f"IS Sharpe: {base_is_sharpe:.3f}" + (f" | OOS Sharpe: {base_oos_sharpe:.3f}" if not np.isnan(base_oos_sharpe) else ""))

    # Define sensitivity ranges for each parameter
    sens_params = {
        'rsi_period':    {'range': rsi_period_range,  'base': base_rp},
        'rsi_oversold':  {'range': rsi_oversold_range, 'base': base_ros},
        'atr_stop_mult': {'range': atr_stop_range,    'base': base_asm},
        'target_mult':   {'range': target_range,      'base': base_tm},
    }

    def eval_sens(rp, ros, asm, tm):
        e_is, x_is = generate_vwap_mr_signals(
            train_close, train_high, train_low, train_open, train_volume,
            rsi_period=rp, rsi_oversold=ros, atr_stop_mult=asm, target_mult=tm)
        pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
        e_oos, x_oos = generate_vwap_mr_signals(
            val_close, val_high, val_low, val_open, val_volume,
            rsi_period=rp, rsi_oversold=ros, atr_stop_mult=asm, target_mult=tm)
        pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                             init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
        return float(pf_is.sharpe_ratio(freq=FREQ)), float(pf_oos.sharpe_ratio(freq=FREQ))

    sens_data = {}
    for pname, pinfo in sens_params.items():
        rows = []
        for val in pinfo['range']:
            params = {'rp': base_rp, 'ros': base_ros, 'asm': base_asm, 'tm': base_tm}
            if pname == 'rsi_period':    params['rp']  = val
            elif pname == 'rsi_oversold': params['ros'] = val
            elif pname == 'atr_stop_mult': params['asm'] = val
            elif pname == 'target_mult':   params['tm']  = val
            try:
                is_s, oos_s = eval_sens(params['rp'], params['ros'], params['asm'], params['tm'])
                is_delta  = (is_s - base_is_sharpe) / abs(base_is_sharpe) * 100 if abs(base_is_sharpe) > 1e-9 else 0
                oos_delta = (oos_s - base_oos_sharpe) / abs(base_oos_sharpe) * 100 if not np.isnan(base_oos_sharpe) and abs(base_oos_sharpe) > 1e-9 else np.nan
                rows.append({'value': val, 'is_sharpe': is_s, 'oos_sharpe': oos_s,
                             'is_delta': is_delta, 'oos_delta': oos_delta})
            except: pass
        sens_data[pname] = pd.DataFrame(rows)

    # Plot 2xN grid
    n_params = len(sens_params)
    fig, axes = plt.subplots(2, n_params, figsize=(5*n_params, 10))
    fig.suptitle(f'Sensitivity - Base: RSI({base_rp}) OS={base_ros} Stop={base_asm}x Target={base_tm}x',
                 fontsize=16, fontweight='bold')

    for ci, (pname, pinfo) in enumerate(sens_params.items()):
        df = sens_data[pname]
        if df.empty: continue
        base_val = pinfo['base']

        # IS row
        colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                  for x in df['is_delta']]
        x_labels = [str(v) for v in df['value']]
        axes[0, ci].bar(x_labels, df['is_delta'], color=colors, edgecolor='black', linewidth=0.5)
        axes[0, ci].axhline(0, color='black', linewidth=1.5)
        axes[0, ci].set_xlabel(pname.replace('_', ' ').title(), fontsize=11, fontweight='bold')
        axes[0, ci].set_ylabel('Sharpe Delta %', fontsize=10)
        axes[0, ci].set_title(f'{pname.replace("_", " ").title()} - In-Sample', fontsize=12, fontweight='bold')
        axes[0, ci].grid(axis='y', alpha=0.3)

        # OOS row
        if not np.isnan(base_oos_sharpe) and 'oos_delta' in df.columns and not df['oos_delta'].isna().all():
            colors2 = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                       for x in df['oos_delta'].fillna(0)]
            axes[1, ci].bar(x_labels, df['oos_delta'], color=colors2, edgecolor='black', linewidth=0.5)
            axes[1, ci].axhline(0, color='black', linewidth=1.5)
            axes[1, ci].set_xlabel(pname.replace('_', ' ').title(), fontsize=11, fontweight='bold')
            axes[1, ci].set_ylabel('Sharpe Delta %', fontsize=10)
            axes[1, ci].set_title(f'{pname.replace("_", " ").title()} - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, ci].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary table
    print("\nSENSITIVITY SUMMARY")
    print("=" * 80)
    summary_rows = []
    for pname, pinfo in sens_params.items():
        df = sens_data[pname]
        if df.empty: continue
        summary_rows.append({
            'Parameter': pname.replace('_', ' ').title(),
            'IS Range': f"{df['is_sharpe'].min():.3f} - {df['is_sharpe'].max():.3f}",
            'IS Max Delta%': f"{df['is_delta'].min():.1f}% to {df['is_delta'].max():.1f}%",
            'OOS Range': f"{df['oos_sharpe'].min():.3f} - {df['oos_sharpe'].max():.3f}" if not df['oos_sharpe'].isna().all() else 'N/A',
            'Sensitivity': 'HIGH' if abs(df['is_delta'].min()) > 40 or abs(df['is_delta'].max()) > 40 else 'LOW',
        })
    print(pd.DataFrame(summary_rows).to_string(index=False))

In [ ]:
# MULTI-ASSET OUT-OF-SAMPLE VALIDATION

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    bp = {
        'rsi_period': int(best['rsi_period']),
        'rsi_oversold': int(best['rsi_oversold']),
        'atr_stop_mult': float(best['atr_stop_mult']),
        'target_mult': float(best['target_mult']),
    }

    print("=" * 90)
    print(f"MULTI-ASSET OOS VALIDATION | Best Params: RSI({bp['rsi_period']}) OS={bp['rsi_oversold']} "
          f"Stop={bp['atr_stop_mult']}x Target={bp['target_mult']}x")
    print("=" * 90)

    multi_results = []

    for ticker in TICKERS:
        try:
            # Extract ticker data from all_data
            if isinstance(all_data.columns, pd.MultiIndex):
                tk_close  = all_data[('Close',  ticker)].astype(float).dropna()
                tk_high   = all_data[('High',   ticker)].astype(float).reindex(tk_close.index)
                tk_low    = all_data[('Low',    ticker)].astype(float).reindex(tk_close.index)
                tk_open   = all_data[('Open',   ticker)].astype(float).reindex(tk_close.index)
                tk_volume = all_data[('Volume', ticker)].astype(float).reindex(tk_close.index)
            else:
                # Single ticker fallback
                tk_close  = all_data['Close'].astype(float).dropna()
                tk_high   = all_data['High'].astype(float).reindex(tk_close.index)
                tk_low    = all_data['Low'].astype(float).reindex(tk_close.index)
                tk_open   = all_data['Open'].astype(float).reindex(tk_close.index)
                tk_volume = all_data['Volume'].astype(float).reindex(tk_close.index)

            # OOS split (use last 40%)
            tk_split = int(len(tk_close) * 0.60)
            tk_oos_close = tk_close.iloc[tk_split:]
            tk_oos_high  = tk_high.iloc[tk_split:]
            tk_oos_low   = tk_low.iloc[tk_split:]
            tk_oos_open  = tk_open.iloc[tk_split:]
            tk_oos_vol   = tk_volume.iloc[tk_split:]

            if len(tk_oos_close) < 50:
                print(f"  {ticker}: insufficient OOS data ({len(tk_oos_close)} bars)")
                continue

            e, x = generate_vwap_mr_signals(
                tk_oos_close, tk_oos_high, tk_oos_low, tk_oos_open, tk_oos_vol,
                rsi_period=bp['rsi_period'], rsi_oversold=bp['rsi_oversold'],
                atr_stop_mult=bp['atr_stop_mult'], target_mult=bp['target_mult'],
                require_trend_filter=True
            )

            pf = vbt.Portfolio.from_signals(
                close=tk_oos_close, entries=e, exits=x,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
            )

            n_trades = len(pf.trades)
            sharpe = float(pf.sharpe_ratio(freq=FREQ)) if n_trades > 0 else np.nan
            ret = float(pf.total_return())
            dd = float(pf.max_drawdown())

            wr = np.nan; pf_val = np.nan
            if n_trades > 0:
                tr_arr = np.asarray(pf.trades.returns.values if hasattr(pf.trades.returns, 'values') else pf.trades.returns).ravel()
                if tr_arr.size > 0:
                    p = tr_arr[tr_arr > 0]; n_arr = tr_arr[tr_arr < 0]
                    wr = (len(p)/len(tr_arr))*100 if len(tr_arr) > 0 else np.nan
                    pf_val = p.sum()/abs(n_arr.sum()) if len(n_arr) > 0 and abs(n_arr.sum()) > 0 else np.inf

            multi_results.append({
                'Ticker': ticker, 'OOS_Sharpe': sharpe, 'OOS_Return': ret,
                'OOS_MaxDD': dd, 'OOS_Trades': n_trades, 'OOS_WinRate': wr,
                'OOS_PF': pf_val, 'OOS_Bars': len(tk_oos_close),
            })

            status = "PASS" if not np.isnan(sharpe) and sharpe > 0 else "FAIL"
            print(f"  {ticker:6s} | Sharpe={sharpe:7.3f} | Return={ret:7.2%} | "
                  f"MaxDD={dd:7.2%} | Trades={n_trades:3d} | WR={wr:5.1f}% | [{status}]")

        except Exception as ex:
            print(f"  {ticker:6s} | ERROR: {ex}")

    if multi_results:
        multi_df = pd.DataFrame(multi_results)
        print(f"\n{'='*90}")
        n_pass = sum(1 for r in multi_results if not np.isnan(r['OOS_Sharpe']) and r['OOS_Sharpe'] > 0)
        print(f"SUMMARY: {n_pass}/{len(multi_results)} tickers profitable OOS")
        print(f"  Avg OOS Sharpe: {multi_df['OOS_Sharpe'].mean():.3f}")
        print(f"  Avg OOS Return: {multi_df['OOS_Return'].mean():.2%}")
        print(f"  Avg OOS MaxDD:  {multi_df['OOS_MaxDD'].mean():.2%}")

In [ ]:
# FTMO MONTE CARLO SIMULATION (10K sims)
# Aggregate hourly returns to daily first, then bootstrap

if results_df.empty:
    print("No results.")
else:
    # Get best strategy's full-sample returns (hourly)
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    bp = {
        'rsi_period': int(best['rsi_period']),
        'rsi_oversold': int(best['rsi_oversold']),
        'atr_stop_mult': float(best['atr_stop_mult']),
        'target_mult': float(best['target_mult']),
    }

    e_full, x_full = generate_vwap_mr_signals(
        close_series, high, low, open_, volume, **bp)
    pf_full = vbt.Portfolio.from_signals(
        close=close_series, entries=e_full, exits=x_full,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

    # Aggregate hourly returns to daily
    hourly_rets = pf_full.returns()
    daily_cum = (1 + hourly_rets).groupby(hourly_rets.index.date).prod() - 1
    dr = daily_cum.values.ravel()
    dr = dr[~np.isnan(dr)]

    print(f"FTMO Monte Carlo Simulation")
    print(f"  Strategy returns: {len(dr)} daily observations (aggregated from hourly)")
    print(f"  Mean daily return: {dr.mean():.4%}")
    print(f"  Daily vol: {dr.std():.4%}")
    print(f"  Running 10,000 simulations of 30 trading days...\n")

    N_SIM = 10_000; DAYS = 30; ACCOUNT = 100_000
    np.random.seed(42)
    n_passed = n_blown_t = n_blown_d = 0
    final_eqs = np.zeros(N_SIM)
    sample_paths = []

    for s in range(N_SIM):
        eq = ACCOUNT
        path = [ACCOUNT]
        sim_rets = np.random.choice(dr, size=DAYS, replace=True)
        blown = False

        for d in range(DAYS):
            day_start = eq
            eq *= (1 + sim_rets[d])
            path.append(eq)

            # Daily loss check: 5% of ACCOUNT
            if (eq - day_start) / ACCOUNT < -0.05:
                n_blown_d += 1; blown = True; break
            # Total loss check: 10% of ACCOUNT
            if (eq - ACCOUNT) / ACCOUNT < -0.10:
                n_blown_t += 1; blown = True; break
            # Profit target: 10% of ACCOUNT
            if (eq - ACCOUNT) / ACCOUNT >= 0.10:
                n_passed += 1; blown = True; break

        while len(path) < DAYS + 1:
            path.append(path[-1])
        final_eqs[s] = path[-1]
        if s < 200:
            sample_paths.append(path)

    n_still = N_SIM - n_passed - n_blown_t - n_blown_d
    pass_rate = n_passed / N_SIM * 100

    verdict = ("FAVORABLE" if pass_rate >= 50 else
               "POSSIBLE" if pass_rate >= 25 else
               "CHALLENGING" if pass_rate >= 10 else "UNLIKELY")

    GREEN = '#059669'; RED = '#DC2626'; ORANGE = '#EA580C'; ACCENT = '#2563EB'
    TEXT_MUT = '#8C95A3'; TEXT_SEC = '#5A6270'
    verdict_color = GREEN if pass_rate >= 50 else ORANGE if pass_rate >= 25 else RED

    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    fig.suptitle(f'FTMO Monte Carlo - {N_SIM:,} Simulations | {TICKER} VWAP Mean Reversion',
                 fontsize=14, fontweight='bold')

    for path in sample_paths:
        c = GREEN if path[-1] >= 110000 else (RED if path[-1] <= 90000 else 'gray')
        ax.plot(range(DAYS+1), path, color=c, alpha=0.12, linewidth=0.5)

    ax.axhline(110000, color=GREEN, linestyle='--', linewidth=2.5, label=f'10% Target ($110k)')
    ax.axhline(90000, color=RED, linestyle='--', linewidth=2.5, label=f'10% Max Loss ($90k)')
    ax.axhline(100000, color='gray', linestyle='-', linewidth=0.8, alpha=0.4)

    ax.set_xlabel('Trading Day', fontsize=11)
    ax.set_ylabel('Equity ($)', fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # Results text box
    results_text = (f"Pass Rate: {pass_rate:.1f}%  |  Verdict: {verdict}\n"
                    f"Passed: {n_passed:,}  |  Blown (total): {n_blown_t:,}  |  "
                    f"Blown (daily): {n_blown_d:,}  |  Still trading: {n_still:,}\n"
                    f"Median Final Equity: ${np.median(final_eqs):,.0f}")
    ax.text(0.5, 0.03, results_text, transform=ax.transAxes, fontsize=10, ha='center',
            family='monospace', bbox=dict(boxstyle='round,pad=0.5', facecolor='#F7F8FA',
                                           edgecolor='#E2E5EB', alpha=0.95))

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*70}")
    print(f"FTMO MONTE CARLO RESULTS")
    print(f"{'='*70}")
    print(f"  Simulations:     {N_SIM:,}")
    print(f"  Pass Rate:       {pass_rate:.1f}%")
    print(f"  Verdict:         {verdict}")
    print(f"  Passed:          {n_passed:,}")
    print(f"  Blown (total):   {n_blown_t:,}")
    print(f"  Blown (daily):   {n_blown_d:,}")
    print(f"  Still Trading:   {n_still:,}")
    print(f"  Median Equity:   ${np.median(final_eqs):,.0f}")
    print(f"{'='*70}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# UNIVERSAL STRATEGY EXPORT — Data Files + PDF Tearsheet
# ════════════════════════════════════════════════════════════════════
# Adapted for VWAP Mean Reversion (hourly/intraday)
# ════════════════════════════════════════════════════════════════════

import os, sys, json, datetime, hashlib, shutil
from matplotlib.backends.backend_pdf import PdfPages

# ═══ EDIT THESE LINES ═══════════════════════════════════════
STRATEGY_NAME = "VWAP_Mean_Reversion"                              # ← EDIT
PARAM_COLS    = ["rsi_period", "rsi_oversold", "atr_stop_mult", "target_mult"]  # ← EDIT
NOTES         = ""                                                   # ← Optional run notes
# ════════════════════════════════════════════════════════════════

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'h'  # hourly

# ── Google Drive mount ──
EXPORT_DIR = "/content/strategy_exports"
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
    IN_COLAB = True
    print("\u2705 Google Drive mounted")
except:
    print("\u26a0\ufe0f Local mode \u2014 exports to ./strategy_exports")

RUN_TIMESTAMP = datetime.datetime.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S")

# ── Folder structure ──
STRAT_DIR   = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
LATEST_DIR  = os.path.join(STRAT_DIR, "latest")
ARCHIVE_DIR = os.path.join(STRAT_DIR, "archive")
os.makedirs(LATEST_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
# Build portfolios for export
# ════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
best_params = {}
for col in PARAM_COLS:
    val = best[col]
    best_params[col] = int(val) if col in ['rsi_period', 'rsi_oversold'] else float(val)
param_str = ", ".join([f"{k}={v}" for k, v in best_params.items()])

full_close = close_series.copy()
full_close.name = 'price'
split_idx = int(len(full_close) * 0.60)
train_c = full_close.iloc[:split_idx].copy()
val_c   = full_close.iloc[split_idx:].copy()

# Full sample
e_full, x_full = generate_vwap_mr_signals(
    full_close, high, low, open_, volume,
    rsi_period=best_params['rsi_period'], rsi_oversold=best_params['rsi_oversold'],
    atr_stop_mult=best_params['atr_stop_mult'], target_mult=best_params['target_mult'])
pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                      init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# IS
train_h = high.iloc[:split_idx]; train_l = low.iloc[:split_idx]
train_o = open_.iloc[:split_idx]; train_v = volume.iloc[:split_idx]
e_is, x_is = generate_vwap_mr_signals(
    train_c, train_h, train_l, train_o, train_v,
    rsi_period=best_params['rsi_period'], rsi_oversold=best_params['rsi_oversold'],
    atr_stop_mult=best_params['atr_stop_mult'], target_mult=best_params['target_mult'])
pf_is = vbt.Portfolio.from_signals(close=train_c, entries=e_is, exits=x_is,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# OOS
val_h = high.iloc[split_idx:]; val_l = low.iloc[split_idx:]
val_o = open_.iloc[split_idx:]; val_v = volume.iloc[split_idx:]
e_oos, x_oos = generate_vwap_mr_signals(
    val_c, val_h, val_l, val_o, val_v,
    rsi_period=best_params['rsi_period'], rsi_oversold=best_params['rsi_oversold'],
    atr_stop_mult=best_params['atr_stop_mult'], target_mult=best_params['target_mult'])
pf_oos = vbt.Portfolio.from_signals(close=val_c, entries=e_oos, exits=x_oos,
                                     init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# Buy & Hold
bh_e = pd.Series(False, index=full_close.index, dtype=bool); bh_e.iloc[0] = True
bh_x = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(close=full_close, entries=bh_e, exits=bh_x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# ── Extract metrics ──
trades_obj = pf_full.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
pos, neg = tr[tr > 0], tr[tr < 0]
years_full = max((full_close.index[-1] - full_close.index[0]).total_seconds() / (365.25 * 86400), 1e-9)
daily_rets_hourly = pf_full.returns()
# Aggregate hourly to daily for export
daily_rets = (1 + daily_rets_hourly).groupby(daily_rets_hourly.index.date).prod() - 1

def safe(fn, default=None):
    try: return float(fn())
    except: return default

M = {
    'total_return': safe(pf_full.total_return), 'ann_return': safe(lambda: pf_full.annualized_return(freq=FREQ)),
    'sharpe': safe(lambda: pf_full.sharpe_ratio(freq=FREQ)), 'sortino': safe(lambda: pf_full.sortino_ratio(freq=FREQ)),
    'max_dd': safe(pf_full.max_drawdown), 'volatility': safe(lambda: pf_full.annualized_volatility(freq=FREQ)),
    'calmar': safe(lambda: pf_full.annualized_return(freq=FREQ)) / abs(safe(pf_full.max_drawdown)) if abs(safe(pf_full.max_drawdown, 0)) > 1e-9 else None,
    'trades': len(trades_obj), 'trades_yr': len(trades_obj) / years_full,
    'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
    'pf': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
    'expectancy': float(tr.mean()) if len(tr) > 0 else None,
    'avg_win': float(pos.mean()) if len(pos) > 0 else None, 'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
    'largest_win': float(pos.max()) if len(pos) > 0 else None, 'largest_loss': float(neg.min()) if len(neg) > 0 else None,
    'payoff': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
    'is_sharpe': safe(lambda: pf_is.sharpe_ratio(freq=FREQ)), 'is_return': safe(pf_is.total_return),
    'is_dd': safe(pf_is.max_drawdown), 'is_trades': len(pf_is.trades),
    'oos_sharpe': safe(lambda: pf_oos.sharpe_ratio(freq=FREQ)), 'oos_return': safe(pf_oos.total_return),
    'oos_dd': safe(pf_oos.max_drawdown), 'oos_trades': len(pf_oos.trades),
    'bh_return': safe(pf_bh.total_return), 'bh_sharpe': safe(lambda: pf_bh.sharpe_ratio(freq=FREQ)),
    'bh_dd': safe(pf_bh.max_drawdown),
}

# ════════════════════════════════════════════════════════════════
# 1. SAVE STRUCTURED DATA FILES
# ════════════════════════════════════════════════════════════════
export_json = {
    "metadata": {
        "run_id": RUN_ID, "export_timestamp": RUN_TIMESTAMP.isoformat(),
        "export_date_human": RUN_TIMESTAMP.strftime("%B %d, %Y at %I:%M %p"),
        "strategy_name": STRATEGY_NAME, "strategy_family": "VWAP",
        "ticker": TICKER,
        "instrument_type": "equity/etf",
        "data_source": "alpaca", "data_interval": "1h", "currency": "USD",
        "start_date": str(full_close.index[0].date()), "end_date": str(full_close.index[-1].date()),
        "total_bars": len(full_close), "total_years": round(years_full, 2),
        "train_start": str(train_c.index[0].date()), "train_end": str(train_c.index[-1].date()),
        "train_bars": len(train_c), "val_start": str(val_c.index[0].date()),
        "val_end": str(val_c.index[-1].date()), "val_bars": len(val_c), "train_ratio": 0.60,
        "init_cash": INIT_CASH, "fees_pct": FEES, "slippage_pct": SLIPPAGE, "frequency": FREQ,
        "first_close": round(float(full_close.iloc[0]), 4), "last_close": round(float(full_close.iloc[-1]), 4),
        "python_version": sys.version.split()[0], "environment": "colab_pro" if IN_COLAB else "local",
        "grid_combos_tested": len(results_df), "param_columns": PARAM_COLS, "notes": NOTES,
    },
    "best_params": best_params,
    "metrics_full_sample": {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
    "metrics_in_sample": {k.replace('is_',''): v for k, v in M.items() if k.startswith('is_')},
    "metrics_out_of_sample": {k.replace('oos_',''): v for k, v in M.items() if k.startswith('oos_')},
    "metrics_buy_hold": {k.replace('bh_',''): v for k, v in M.items() if k.startswith('bh_')},
    "grid_search_summary": {
        "top5": results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio','total_return','max_drawdown']].to_dict('records'),
    }
}

# Save JSON
with open(os.path.join(LATEST_DIR, "summary.json"), 'w', encoding='utf-8') as f:
    json.dump(export_json, f, indent=2, default=str)
with open(os.path.join(ARCHIVE_DIR, f"{RUN_ID}_summary.json"), 'w', encoding='utf-8') as f:
    json.dump(export_json, f, indent=2, default=str)
print(f"\u2705 summary.json")

# Save CSVs
daily_dates = daily_rets.index
pd.DataFrame({'date': daily_dates, 'strategy_return': daily_rets.values,
              'portfolio_value': pf_full.value().groupby(pf_full.value().index.date).last().values[:len(daily_dates)]
}).to_csv(os.path.join(LATEST_DIR, "daily_returns.csv"), index=False, encoding='utf-8')
print(f"\u2705 daily_returns.csv")

pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl,
              'cumulative_pnl': np.cumsum(pnl), 'is_winner': tr > 0
}).to_csv(os.path.join(LATEST_DIR, "trades.csv"), index=False, encoding='utf-8')
print(f"\u2705 trades.csv")

results_df.to_csv(os.path.join(LATEST_DIR, "grid_results.csv"), index=False, encoding='utf-8')
print(f"\u2705 grid_results.csv")

# Run log
log_path = os.path.join(EXPORT_DIR, "run_log.csv")
log_entry = pd.DataFrame([{
    "run_id": RUN_ID, "timestamp": RUN_TIMESTAMP.isoformat(), "strategy": STRATEGY_NAME,
    "ticker": TICKER, "best_params": str(best_params),
    "sharpe_full": round(M['sharpe'] or 0, 4), "sharpe_is": round(M['is_sharpe'] or 0, 4),
    "sharpe_oos": round(M['oos_sharpe'] or 0, 4), "total_return": round(M['total_return'] or 0, 4),
    "max_drawdown": round(M['max_dd'] or 0, 4), "total_trades": M['trades'],
    "win_rate": round(M['win_rate'] or 0, 1), "profit_factor": round(M['pf'] or 0, 2) if M['pf'] else None,
    "notes": NOTES, "export_path": STRAT_DIR,
}])
if os.path.exists(log_path):
    log_combined = pd.concat([pd.read_csv(log_path), log_entry], ignore_index=True)
else:
    log_combined = log_entry
log_combined.to_csv(log_path, index=False, encoding='utf-8')
print(f"\u2705 run_log.csv ({len(log_combined)} runs)")

# ════════════════════════════════════════════════════════════════
# 2. GENERATE PDF TEARSHEET
# ════════════════════════════════════════════════════════════════
pdf_path = os.path.join(LATEST_DIR, "tearsheet.pdf")
pdf_archive = os.path.join(ARCHIVE_DIR, f"{RUN_ID}_tearsheet.pdf")

fmt = lambda v, d=2: f"{v:.{d}f}" if v is not None and not np.isnan(v) else "N/A"
fmtp = lambda v: f"{v:.2%}" if v is not None and not np.isnan(v) else "N/A"

BG       = '#FFFFFF'
CARD_BG  = '#F7F8FA'
CARD_BRD = '#E2E5EB'
TEXT_PRI = '#1A1D23'
TEXT_SEC = '#5A6270'
TEXT_MUT = '#8C95A3'
ACCENT   = '#2563EB'
GREEN    = '#059669'
RED      = '#DC2626'
ORANGE   = '#EA580C'
GRID_CLR = '#E5E7EB'

def _draw_card(ax_fig, x, y, w, h, label, value, color=ACCENT, fontsize_val=22):
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                           facecolor=CARD_BG, edgecolor=CARD_BRD, linewidth=1.2,
                           transform=ax_fig.transAxes, zorder=2)
    ax_fig.add_patch(rect)
    ax_fig.text(x + w/2, y + h*0.68, value, ha='center', va='center',
                fontsize=fontsize_val, fontweight='bold', color=color,
                transform=ax_fig.transAxes, zorder=3)
    ax_fig.text(x + w/2, y + h*0.25, label, ha='center', va='center',
                fontsize=8, color=TEXT_SEC, transform=ax_fig.transAxes, zorder=3)

def _style_ax(ax, title=None):
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT_SEC, labelsize=8)
    ax.grid(True, alpha=0.4, color=GRID_CLR, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_color(CARD_BRD); spine.set_linewidth(0.8)
    if title:
        ax.set_title(title, color=TEXT_PRI, fontsize=11, fontweight='bold', pad=10)

with PdfPages(pdf_path) as pdf:

    # ── PAGE 1: Dashboard ──
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    from matplotlib.patches import FancyBboxPatch, Rectangle
    header = Rectangle((0, 0.91), 1, 0.09, facecolor=ACCENT, transform=ax.transAxes, zorder=1)
    ax.add_patch(header)
    ax.text(0.04, 0.955, "STRATEGY TEARSHEET", ha='left', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes, zorder=2)
    ax.text(0.96, 0.955, f"{STRATEGY_NAME}  |  {TICKER}", ha='right', va='center',
            fontsize=12, color='rgba(255,255,255,0.85)', transform=ax.transAxes, zorder=2, family='monospace')
    ax.text(0.96, 0.92, f"{full_close.index[0].date()} to {full_close.index[-1].date()}  |  {len(full_close)} bars  |  {param_str}",
            ha='right', va='center', fontsize=8, color='rgba(255,255,255,0.65)', transform=ax.transAxes, zorder=2)

    card_w, card_h = 0.145, 0.09; card_y = 0.79
    cards = [
        ("SHARPE", fmt(M['sharpe'], 3), ACCENT),
        ("RETURN", fmtp(M['total_return']), GREEN if (M['total_return'] or 0) > 0 else RED),
        ("MAX DD", fmtp(M['max_dd']), RED if (M['max_dd'] or 0) < -0.15 else ORANGE),
        ("WIN RATE", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", GREEN if (M['win_rate'] or 0) > 50 else RED),
        ("TRADES", str(M['trades']), TEXT_PRI),
        ("PROFIT FACTOR", fmt(M['pf']), GREEN if (M['pf'] or 0) > 1.5 else (ORANGE if (M['pf'] or 0) > 1 else RED)),
    ]
    x_start = 0.03; gap = (0.94 - len(cards) * card_w) / (len(cards) - 1)
    for i, (label, value, color) in enumerate(cards):
        cx = x_start + i * (card_w + gap)
        _draw_card(ax, cx, card_y, card_w, card_h, label, value, color)

    rows = [
        ["METRIC", "FULL SAMPLE", "IN-SAMPLE", "OUT-OF-SAMPLE", "BUY & HOLD"],
        ["Total Return", fmtp(M['total_return']), fmtp(M['is_return']), fmtp(M['oos_return']), fmtp(M['bh_return'])],
        ["Sharpe Ratio", fmt(M['sharpe'], 3), fmt(M['is_sharpe'], 3), fmt(M['oos_sharpe'], 3), fmt(M['bh_sharpe'], 3)],
        ["Sortino Ratio", fmt(M['sortino'], 3), "\u2014", "\u2014", "\u2014"],
        ["Max Drawdown", fmtp(M['max_dd']), fmtp(M['is_dd']), fmtp(M['oos_dd']), fmtp(M['bh_dd'])],
        ["Calmar Ratio", fmt(M['calmar'], 3), "\u2014", "\u2014", "\u2014"],
        ["Volatility (Ann.)", fmtp(M['volatility']), "\u2014", "\u2014", "\u2014"],
        ["Win Rate", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", "\u2014", "\u2014", "\u2014"],
        ["Profit Factor", fmt(M['pf']), "\u2014", "\u2014", "\u2014"],
        ["Expectancy", fmt(M['expectancy'], 4), "\u2014", "\u2014", "\u2014"],
        ["Payoff Ratio", fmt(M['payoff']), "\u2014", "\u2014", "\u2014"],
        ["Total Trades", str(M['trades']), str(M['is_trades']), str(M['oos_trades']), "1"],
        ["Trades/Year", fmt(M['trades_yr'], 1), "\u2014", "\u2014", "\u2014"],
        ["Avg Win", fmtp(M['avg_win']), "\u2014", "\u2014", "\u2014"],
        ["Avg Loss", fmtp(M['avg_loss']), "\u2014", "\u2014", "\u2014"],
        ["Largest Win", fmtp(M['largest_win']), "\u2014", "\u2014", "\u2014"],
        ["Largest Loss", fmtp(M['largest_loss']), "\u2014", "\u2014", "\u2014"],
    ]
    table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
                     bbox=[0.03, 0.04, 0.94, 0.70])
    table.auto_set_font_size(False); table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(CARD_BRD); cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor(ACCENT); cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor(BG if row % 2 == 1 else CARD_BG)
            cell.set_text_props(color=TEXT_PRI, fontsize=8, family='monospace')
            if col == 0: cell.set_text_props(color=TEXT_PRI, fontsize=8, fontweight='bold')
        cell.set_height(0.052)

    ax.text(0.5, 0.01, f"Run {RUN_ID}  |  QS Finance", ha='center', va='bottom',
            fontsize=7, color=TEXT_MUT, transform=ax.transAxes)
    pdf.savefig(fig, facecolor=BG); plt.close(fig)

    # ── PAGE 2: Equity Curve + Drawdown ──
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor(BG)
    fig.suptitle(f'{STRATEGY_NAME} on {TICKER} \u2014 Equity & Drawdown', fontsize=14,
                 fontweight='bold', color=TEXT_PRI, y=0.97)
    eq_strat = pf_full.value(); eq_bh = pf_bh.value()
    _style_ax(ax1)
    ax1.plot(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values,
             color=ACCENT, linewidth=2, label='Strategy (IS)', solid_capstyle='round')
    ax1.plot(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values,
             color=ORANGE, linewidth=2, label='Strategy (OOS)', solid_capstyle='round')
    ax1.plot(full_close.index, eq_bh.values, color=TEXT_MUT, linewidth=1.2, alpha=0.6, linestyle='--', label='Buy & Hold')
    ax1.axvline(x=full_close.index[split_idx], color=RED, linestyle=':', alpha=0.3, linewidth=1)
    ax1.set_ylabel('Portfolio Value ($)', color=TEXT_SEC, fontsize=9)
    ax1.legend(fontsize=8, facecolor=BG, edgecolor=CARD_BRD)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    stats_text = f"Sharpe: {fmt(M['sharpe'],3)}   |   Return: {fmtp(M['total_return'])}   |   MaxDD: {fmtp(M['max_dd'])}   |   WR: {M['win_rate']:.1f}%   |   PF: {fmt(M['pf'])}"
    ax1.text(0.5, 0.03, stats_text, transform=ax1.transAxes, fontsize=8, ha='center', color=TEXT_SEC, family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))
    _style_ax(ax2)
    dd = pf_full.drawdown() * 100
    ax2.fill_between(full_close.index, dd.values, 0, color=RED, alpha=0.2)
    ax2.plot(full_close.index, dd.values, color=RED, linewidth=0.8, alpha=0.7)
    ax2.set_ylabel('Drawdown %', color=TEXT_SEC, fontsize=9); ax2.set_xlabel('Date', color=TEXT_SEC, fontsize=9)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig, facecolor=BG); plt.close(fig)

    # ── PAGE 3: Trade Analysis ──
    if len(tr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5)); fig.patch.set_facecolor(BG)
        fig.suptitle(f'Trade-by-Trade Analysis \u2014 {len(tr)} Trades', fontsize=14, fontweight='bold', color=TEXT_PRI, y=0.97)
        n = len(tr); colors_bar = [GREEN if r > 0 else RED for r in tr]; colors_pnl = [GREEN if p > 0 else RED for p in pnl]
        for a in axes.flat: _style_ax(a)
        axes[0,0].bar(range(n), tr*100, color=colors_bar, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,0].axhline(np.mean(tr)*100, color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: {np.mean(tr)*100:.2f}%')
        axes[0,0].axhline(0, color=TEXT_MUT, linewidth=0.5); axes[0,0].set_title('Trade Returns (%)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,0].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)
        axes[0,1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,1].axhline(np.mean(pnl), color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: ${np.mean(pnl):,.0f}')
        axes[0,1].axhline(0, color=TEXT_MUT, linewidth=0.5); axes[0,1].set_title('Trade P&L ($)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}')); axes[0,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)
        cum_pnl = np.cumsum(pnl)
        axes[1,0].plot(range(1, n+1), cum_pnl, color=ACCENT, linewidth=2.5, solid_capstyle='round')
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl>=0, alpha=0.12, color=GREEN)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl<0, alpha=0.12, color=RED)
        axes[1,0].axhline(0, color=TEXT_MUT, linewidth=0.5); axes[1,0].set_title('Cumulative P&L', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[1,1].hist(tr*100, bins=min(30, max(10, n//3)), color=ACCENT, edgecolor='white', alpha=0.75, linewidth=0.5)
        axes[1,1].axvline(np.mean(tr)*100, color=RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[1,1].axvline(0, color=TEXT_MUT, linewidth=0.8, alpha=0.5); axes[1,1].set_title('Return Distribution', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)
        plt.tight_layout(rect=[0, 0, 1, 0.95]); pdf.savefig(fig, facecolor=BG); plt.close(fig)

    # ── PAGE 4: Monte Carlo FTMO ──
    dr_mc = daily_rets.values.ravel(); dr_mc = dr_mc[~np.isnan(dr_mc)]
    N_SIM = 5000; DAYS = 30; ACCOUNT = 100000
    np.random.seed(42)
    n_passed_mc = n_blown_t_mc = n_blown_d_mc = 0
    final_eqs_mc = np.zeros(N_SIM); sample_paths_mc = []
    for s in range(N_SIM):
        eq = ACCOUNT; path = [ACCOUNT]
        sim_rets = np.random.choice(dr_mc, size=DAYS, replace=True)
        for d in range(DAYS):
            day_start = eq; eq *= (1 + sim_rets[d]); path.append(eq)
            if (eq - day_start)/ACCOUNT < -0.05: n_blown_d_mc += 1; break
            if (eq - ACCOUNT)/ACCOUNT < -0.10: n_blown_t_mc += 1; break
            if (eq - ACCOUNT)/ACCOUNT >= 0.10: n_passed_mc += 1; break
        while len(path) < DAYS + 1: path.append(path[-1])
        final_eqs_mc[s] = path[-1]
        if s < 150: sample_paths_mc.append(path)
    n_still_mc = N_SIM - n_passed_mc - n_blown_t_mc - n_blown_d_mc
    pass_rate_mc = n_passed_mc / N_SIM * 100
    verdict_mc = "FAVORABLE" if pass_rate_mc >= 50 else "POSSIBLE" if pass_rate_mc >= 25 else "CHALLENGING" if pass_rate_mc >= 10 else "UNLIKELY"
    verdict_color_mc = GREEN if pass_rate_mc >= 50 else ORANGE if pass_rate_mc >= 25 else RED

    fig = plt.figure(figsize=(11, 8.5)); fig.patch.set_facecolor(BG)
    ax_top = fig.add_axes([0, 0.82, 1, 0.18]); ax_top.set_xlim(0, 1); ax_top.set_ylim(0, 1); ax_top.axis('off')
    ax_top.text(0.5, 0.85, f'FTMO Monte Carlo \u2014 {N_SIM:,} Simulations', ha='center', va='center',
                fontsize=16, fontweight='bold', color=TEXT_PRI, transform=ax_top.transAxes)
    mc_cards = [("PASS RATE", f"{pass_rate_mc:.1f}%", verdict_color_mc), ("VERDICT", verdict_mc, verdict_color_mc),
                ("PASSED", f"{n_passed_mc:,}", GREEN), ("BLOWN (TOTAL)", f"{n_blown_t_mc:,}", RED),
                ("BLOWN (DAILY)", f"{n_blown_d_mc:,}", RED), ("STILL TRADING", f"{n_still_mc:,}", TEXT_SEC)]
    mc_cw = 0.14; mc_gap = (0.92 - len(mc_cards) * mc_cw) / (len(mc_cards) - 1)
    for i, (label, value, color) in enumerate(mc_cards):
        cx = 0.04 + i * (mc_cw + mc_gap)
        _draw_card(ax_top, cx, 0.05, mc_cw, 0.65, label, value, color, fontsize_val=16)
    ax_mc = fig.add_axes([0.08, 0.08, 0.86, 0.70]); _style_ax(ax_mc)
    for path in sample_paths_mc:
        c = GREEN if path[-1] >= 110000 else (RED if path[-1] <= 90000 else TEXT_MUT)
        ax_mc.plot(range(DAYS+1), path, color=c, alpha=0.15, linewidth=0.5)
    ax_mc.axhline(110000, color=GREEN, linestyle='--', linewidth=2.5, label='10% Target ($110k)')
    ax_mc.axhline(90000, color=RED, linestyle='--', linewidth=2.5, label='10% Max Loss ($90k)')
    ax_mc.axhline(100000, color=TEXT_MUT, linestyle='-', linewidth=0.8, alpha=0.4)
    ax_mc.set_xlabel('Trading Day', color=TEXT_SEC, fontsize=9); ax_mc.set_ylabel('Equity ($)', color=TEXT_SEC, fontsize=9)
    ax_mc.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax_mc.legend(fontsize=9, facecolor=BG, edgecolor=CARD_BRD)
    ax_mc.text(0.5, 0.03, f"Median Final Equity: ${np.median(final_eqs_mc):,.0f}", transform=ax_mc.transAxes,
               fontsize=9, ha='center', color=TEXT_SEC, family='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))
    pdf.savefig(fig, facecolor=BG); plt.close(fig)

shutil.copy2(pdf_path, pdf_archive)

# Add MC to JSON
mc_data = {'pass_rate': round(pass_rate_mc, 1), 'n_simulations': N_SIM, 'n_passed': n_passed_mc,
           'n_blown_total': n_blown_t_mc, 'n_blown_daily': n_blown_d_mc, 'n_still_trading': n_still_mc,
           'median_final_equity': round(float(np.median(final_eqs_mc)), 2), 'verdict': verdict_mc}
export_json['monte_carlo_ftmo'] = mc_data
with open(os.path.join(LATEST_DIR, 'summary.json'), 'w', encoding='utf-8') as f:
    json.dump(export_json, f, indent=2, default=str)

# ════════════════════════════════════════════════════════════════
# LOG TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════════
try:
    from lib.sheets_logger import SheetsLogger
    _logger = SheetsLogger()
    _logger.log_result(export_json)
    _trades_path = os.path.join(LATEST_DIR, "trades.csv")
    if os.path.exists(_trades_path):
        _trades_df = pd.read_csv(_trades_path)
        _logger.log_trades(TICKER, STRATEGY_NAME, RUN_ID, _trades_df)
except Exception as _e:
    print(f"\u26a0\ufe0f Sheets logging skipped: {_e}")

# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"\U0001f4e6 EXPORT COMPLETE \u2014 {STRATEGY_NAME} on {TICKER}")
print(f"{'='*70}")
print(f"  Run ID:       {RUN_ID}")
print(f"  Timestamp:    {RUN_TIMESTAMP.strftime('%B %d, %Y at %I:%M:%S %p')}")
print(f"  Instrument:   {TICKER} ({export_json['metadata']['instrument_type']})")
print(f"  Params:       {param_str}")
print(f"  Sharpe:       {fmt(M['sharpe'],3)} (IS: {fmt(M['is_sharpe'],3)} -> OOS: {fmt(M['oos_sharpe'],3)})")
print(f"  FTMO Verdict: {verdict_mc} ({pass_rate_mc:.1f}% pass rate)")
print(f"{'='*70}")
print(f"\n\U0001f4c2 {STRAT_DIR}/")
print(f"  \u251c\u2500\u2500 latest/")
print(f"  \u2502   \u251c\u2500\u2500 summary.json")
print(f"  \u2502   \u251c\u2500\u2500 daily_returns.csv")
print(f"  \u2502   \u251c\u2500\u2500 trades.csv")
print(f"  \u2502   \u251c\u2500\u2500 grid_results.csv")
print(f"  \u2502   \u2514\u2500\u2500 tearsheet.pdf")
print(f"  \u2514\u2500\u2500 archive/")
print(f"      \u251c\u2500\u2500 {RUN_ID}_summary.json")
print(f"      \u2514\u2500\u2500 {RUN_ID}_tearsheet.pdf")
print(f"\n\U0001f4cb run_log.csv ({len(log_combined)} total runs)")